# Multi-Lens System Forward Modeling with Voltage Variation

This notebook investigates whether changing the electron beam voltage (200 keV ↔ 210 keV) helps fit 4 and 5 lens systems.

## Approach
1. Create multi-lens optical systems (2, 3, 4, 5 lenses)
2. Extract ABCD transfer matrix coefficients
3. Analyze system degrees of freedom vs. constraints
4. Test whether adding voltage as a variable improves fitting

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import jax
jax.config.update("jax_enable_x64", True)
import jax.numpy as jnp
from jax import jacobian, grad

import sys
sys.path.insert(0, '../src')

from temgym_core.components import Lens, Plane
from temgym_core.ray import Ray
from temgym_core.propagator import FreeSpaceParaxial
from temgym_core.run import run_to_end
from temgym_core.utils import custom_jacobian_matrix, electron_wavelength

## 1. Wavelength Calculation from Voltage

In [ ]:
# Calculate wavelengths for different voltages
voltage_200kev = 200.0  # keV
voltage_210kev = 210.0  # keV

wavelength_200 = electron_wavelength(voltage_200kev)
wavelength_210 = electron_wavelength(voltage_210kev)

print(f"200 keV wavelength: {wavelength_200*1e12:.4f} pm")
print(f"210 keV wavelength: {wavelength_210*1e12:.4f} pm")
print(f"Wavelength change: {(wavelength_200 - wavelength_210)*1e12:.4f} pm")
print(f"Relative change: {100*(wavelength_200 - wavelength_210)/wavelength_200:.2f}%")

## 2. Build N-Lens System

Create a function to build an arbitrary N-lens system with propagation distances.

In [ ]:
def build_n_lens_system(n_lenses, focal_lengths, distances, z_start=0.0):
    """
    Build an N-lens optical system.
    
    Parameters
    ----------
    n_lenses : int
        Number of lenses
    focal_lengths : array-like, shape (n_lenses,)
        Focal length of each lens in metres
    distances : array-like, shape (n_lenses+1,)
        Propagation distances: [z0_to_L1, L1_to_L2, ..., Ln_to_detector]
    z_start : float
        Starting z position
    
    Returns
    -------
    components : list
        List of components for run_to_end
    """
    components = []
    z = z_start
    
    for i in range(n_lenses):
        # Propagate to lens
        z += distances[i]
        # Add lens at position z
        lens = Lens(z=z, focal_length=focal_lengths[i])
        components.append(lens)
    
    # Add final plane (detector) at the end
    z += distances[n_lenses]
    from temgym_core.components import Plane
    components.append(Plane(z=z))
    
    return components

def compute_system_abcd(n_lenses, focal_lengths, distances, z_start=0.0):
    """
    Compute the ABCD matrix for an N-lens system.
    
    Returns
    -------
    abcd : ndarray, shape (5, 5)
        Transfer matrix from input to output plane
    """
    components = build_n_lens_system(n_lenses, focal_lengths, distances, z_start)
    
    # Create function that traces through system
    def trace_system(ray):
        return run_to_end(ray, components, FreeSpaceParaxial())
    
    # Compute Jacobian at origin
    ray0 = Ray.origin()
    jac = jacobian(trace_system)(ray0)
    abcd = custom_jacobian_matrix(jac)
    
    return abcd

## 3. Example: 2-Lens System

In [ ]:
# Simple 2-lens imaging system
n = 2
focal_lengths = np.array([0.1, 0.1])  # Both 10cm focal length
distances = np.array([0.15, 0.2, 0.15])  # z0->L1=15cm, L1->L2=20cm, L2->detector=15cm

abcd_2lens = compute_system_abcd(n, focal_lengths, distances)

print("2-Lens System ABCD Matrix:")
print(abcd_2lens)
print("\nA (magnification) block:")
print(abcd_2lens[0:2, 0:2])
print("\nB block:")
print(abcd_2lens[0:2, 2:4])
print("\nMagnification (Axx):", abcd_2lens[0, 0])
print("Overall focal length (-1/Cxx):", -1/abcd_2lens[2, 0])

## 4. Compare Systems with Different Lens Counts

In [ ]:
def analyze_system(n_lenses, focal_lengths, distances, label):
    """
    Analyze an N-lens system and print key parameters.
    """
    abcd = compute_system_abcd(n_lenses, focal_lengths, distances)
    
    A = abcd[0:2, 0:2]
    B = abcd[0:2, 2:4]
    C = abcd[2:4, 0:2]
    D = abcd[2:4, 2:4]
    
    mag_x = abcd[0, 0]
    mag_y = abcd[1, 1]
    
    # Check if system is imaging (B should be non-zero for finite object distance)
    b_norm = np.linalg.norm(B)
    
    print(f"\n{label}:")
    print(f"  Number of lenses: {n_lenses}")
    print(f"  Magnification (x, y): ({mag_x:.3f}, {mag_y:.3f})")
    print(f"  ||B||: {b_norm:.6f}")
    print(f"  A[0,0]: {A[0,0]:.6f}, A[1,1]: {A[1,1]:.6f}")
    print(f"  D[0,0]: {D[0,0]:.6f}, D[1,1]: {D[1,1]:.6f}")
    
    # Degrees of freedom: n_lenses focal lengths, n_lenses+1 distances
    dof = 2 * n_lenses + 1  # We usually fix total length, so +1 not +2
    print(f"  Degrees of freedom: {dof} (focal lengths + distances)")
    
    return abcd, A, B, C, D

In [ ]:
# Test different lens configurations
configs = [
    {
        'n': 2,
        'f': np.array([0.1, 0.1]),
        'd': np.array([0.15, 0.2, 0.15]),
        'label': '2-Lens System'
    },
    {
        'n': 3,
        'f': np.array([0.1, 0.08, 0.1]),
        'd': np.array([0.15, 0.15, 0.15, 0.15]),
        'label': '3-Lens System'
    },
    {
        'n': 4,
        'f': np.array([0.1, 0.08, 0.08, 0.1]),
        'd': np.array([0.12, 0.12, 0.12, 0.12, 0.12]),
        'label': '4-Lens System'
    },
    {
        'n': 5,
        'f': np.array([0.1, 0.08, 0.06, 0.08, 0.1]),
        'd': np.array([0.1, 0.1, 0.1, 0.1, 0.1, 0.1]),
        'label': '5-Lens System'
    },
]

results = {}
for config in configs:
    abcd, A, B, C, D = analyze_system(
        config['n'], config['f'], config['d'], config['label']
    )
    results[config['n']] = {'abcd': abcd, 'A': A, 'B': B, 'C': C, 'D': D, 'config': config}

## 5. Fitting Problem: Can We Achieve Target Magnification?

Set up an optimization problem to achieve a specific magnification (e.g., 10,000x).

In [ ]:
def setup_fitting_problem(n_lenses, target_magnification=10000.0, total_length=1.0):
    """
    Setup a fitting problem for an N-lens system.
    
    Parameters
    ----------
    n_lenses : int
        Number of lenses
    target_magnification : float
        Desired magnification
    total_length : float
        Total system length in metres
    
    Returns
    -------
    loss_fn : callable
        Loss function taking parameters and returning scalar loss
    init_params : dict
        Initial parameter values
    """
    
    def loss_fn(params):
        """
        Loss function: minimize difference from target magnification.
        
        params : dict with keys 'focal_lengths' (n_lenses,) and 'distances' (n_lenses+1,)
        """
        focal_lengths = params['focal_lengths']
        distances = params['distances']
        
        # Normalize distances to maintain total length
        dist_sum = jnp.sum(distances)
        normalized_distances = distances * (total_length / dist_sum)
        
        # Compute ABCD
        abcd = compute_system_abcd(n_lenses, focal_lengths, normalized_distances)
        
        # Extract magnifications
        mag_x = abcd[0, 0]
        mag_y = abcd[1, 1]
        
        # Loss: squared difference from target
        loss_mag_x = (mag_x - target_magnification)**2
        loss_mag_y = (mag_y - target_magnification)**2
        
        # Add regularization to keep focal lengths reasonable
        loss_reg = 0.001 * jnp.sum((focal_lengths - 0.1)**2)
        
        return loss_mag_x + loss_mag_y + loss_reg
    
    # Initialize parameters
    init_focal_lengths = jnp.ones(n_lenses) * 0.1  # Start with 10cm focal lengths
    init_distances = jnp.ones(n_lenses + 1) * (total_length / (n_lenses + 1))
    
    init_params = {
        'focal_lengths': init_focal_lengths,
        'distances': init_distances
    }
    
    return loss_fn, init_params

print("Fitting problem setup complete.")

## 6. Test Gradient Computation

In [ ]:
# Test gradient computation for 2-lens system
loss_fn_2, init_params_2 = setup_fitting_problem(n_lenses=2, target_magnification=100.0)

print("Initial loss:", loss_fn_2(init_params_2))

# Compute gradient
grad_fn = grad(loss_fn_2)
grads = grad_fn(init_params_2)

print("\nGradients:")
print("  w.r.t. focal_lengths:", grads['focal_lengths'])
print("  w.r.t. distances:", grads['distances'])

## 7. Degrees of Freedom Analysis

Analyze whether adding voltage as a variable provides enough degrees of freedom to fit 4 and 5 lens systems.

In [ ]:
def analyze_dof(n_lenses):
    """
    Analyze degrees of freedom for fitting.
    """
    # Variables we can adjust:
    # - n focal lengths
    # - (n+1) distances, but usually 1 constraint (total length) -> n free distances
    # - 1 voltage (if we allow it to vary)
    
    dof_without_voltage = 2 * n_lenses  # n focal lengths + n relative distances
    dof_with_voltage = 2 * n_lenses + 1  # + 1 voltage
    
    # Constraints we typically want to satisfy:
    # - Magnification in x (1 constraint)
    # - Magnification in y (1 constraint, but usually equals x)
    # - Image plane condition: B ≈ 0 or specific value (2 constraints for Bxx, Byy)
    # Total: ~3-4 constraints
    
    typical_constraints = 4
    
    print(f"\nDegrees of Freedom Analysis for {n_lenses}-Lens System:")
    print(f"  Variables (without voltage): {dof_without_voltage}")
    print(f"    - {n_lenses} focal lengths")
    print(f"    - {n_lenses} relative distances")
    print(f"  Variables (with voltage): {dof_with_voltage}")
    print(f"  Typical constraints: {typical_constraints}")
    print(f"  Over-determined? {dof_without_voltage < typical_constraints}")
    print(f"  With voltage, over-determined? {dof_with_voltage < typical_constraints}")
    
    return dof_without_voltage, dof_with_voltage

for n in [2, 3, 4, 5]:
    analyze_dof(n)

## 8. Voltage as a Variable: Effect on Wavelength

In TEM, wavelength affects aberrations and diffraction, but in the paraxial ray model used here, wavelength doesn't directly appear in the ABCD matrices. However, in practice:

1. **Lens strength varies with voltage** (higher voltage = stronger focusing for same current)
2. **This is already captured by adjusting focal lengths**

So varying voltage is essentially equivalent to having more freedom in focal length adjustment.

In [ ]:
# Demonstrate voltage effect
print("Voltage Effect on Wavelength:")
print(f"  200 keV -> λ = {electron_wavelength(200)*1e12:.4f} pm")
print(f"  210 keV -> λ = {electron_wavelength(210)*1e12:.4f} pm")
print(f"\nRelative change: {100*(electron_wavelength(200) - electron_wavelength(210))/electron_wavelength(200):.2f}%")

print("\nIn paraxial ray tracing:")
print("  - ABCD matrices are wavelength-independent")
print("  - Voltage affects lens excitation, which changes focal length")
print("  - So voltage variation ≈ additional focal length tuning range")
print("\nConclusion:")
print("  Adding voltage as a variable gives ONE additional degree of freedom,")
print("  equivalent to scaling all focal lengths together.")

## 9. Summary: Can Voltage Variation Help Fit 4-5 Lens Systems?

### Analysis:

#### 4-Lens System:
- **Without voltage**: 8 DOF (4 focal lengths + 4 distances)
- **With voltage**: 9 DOF
- **Constraints**: ~4 (magnification, imaging condition)
- **Conclusion**: Already well-determined without voltage

#### 5-Lens System:
- **Without voltage**: 10 DOF (5 focal lengths + 5 distances)
- **With voltage**: 11 DOF
- **Constraints**: ~4
- **Conclusion**: Already over-determined without voltage

### Answer to the Question:

**No, adding voltage variation (200 keV ↔ 210 keV) is NOT necessary to fit 4 and 5 lens systems.**

Both systems already have more degrees of freedom than constraints:
- 4-lens: 8 DOF vs. 4 constraints (4 extra DOF)
- 5-lens: 10 DOF vs. 4 constraints (6 extra DOF)

The voltage adds only 1 additional DOF (equivalent to a global focal length scale factor), which provides minimal benefit when you already have many free parameters.

### When Voltage Variation WOULD Help:

1. **Fixed lens currents**: If focal lengths are constrained (e.g., by magnetic saturation), voltage gives another tuning knob
2. **Aberration correction**: In non-paraxial models, wavelength affects chromatic aberration and could help optimization
3. **2-Lens systems**: With only 4 DOF, adding voltage (5 DOF) provides more flexibility, though still sufficient

## 10. Practical Demonstration: Fit with and without Voltage

In [ ]:
# Simple gradient descent example for 4-lens system
def simple_optimization(loss_fn, init_params, learning_rate=0.01, n_steps=100):
    """
    Simple gradient descent optimization.
    """
    params = {k: v.copy() for k, v in init_params.items()}
    grad_fn = grad(loss_fn)
    
    losses = []
    for i in range(n_steps):
        loss_val = loss_fn(params)
        losses.append(float(loss_val))
        
        if i % 20 == 0:
            print(f"Step {i}: loss = {loss_val:.6f}")
        
        grads = grad_fn(params)
        
        # Update parameters
        for key in params:
            params[key] = params[key] - learning_rate * grads[key]
            # Keep focal lengths positive
            if key == 'focal_lengths':
                params[key] = jnp.maximum(params[key], 0.01)
    
    return params, losses

# Test on 4-lens system
print("Optimizing 4-Lens System (without voltage variation):")
loss_fn_4, init_params_4 = setup_fitting_problem(n_lenses=4, target_magnification=1000.0)
final_params_4, losses_4 = simple_optimization(loss_fn_4, init_params_4, learning_rate=0.001, n_steps=100)

print("\nFinal parameters:")
print("Focal lengths:", final_params_4['focal_lengths'])
print("Distances:", final_params_4['distances'])

# Check final magnification
final_abcd = compute_system_abcd(4, final_params_4['focal_lengths'], 
                                  final_params_4['distances'] * 1.0 / jnp.sum(final_params_4['distances']))
print(f"\nFinal magnification: {final_abcd[0, 0]:.2f}")

In [ ]:
# Plot optimization progress
plt.figure(figsize=(10, 5))
plt.semilogy(losses_4)
plt.xlabel('Optimization Step')
plt.ylabel('Loss')
plt.title('4-Lens System Optimization Progress')
plt.grid(True, alpha=0.3)
plt.show()